In [ ]:
import os
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from transformers import GLPNForDepthEstimation, GLPNImageProcessor
from tqdm import tqdm
import matplotlib.pyplot as plt
import random

# ==========================
# DEVICE ET BATCH
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 4

# ==========================
# DATASET TEST
# ==========================
class DepthDatasetTest(Dataset):
    def __init__(self, rgb_dir, depth_dir=None):
        self.rgb_files = sorted([os.path.join(rgb_dir, f) for f in os.listdir(rgb_dir)
                                 if f.endswith(('.png', '.jpg', '.jpeg'))])
        self.depth_files = None
        if depth_dir:
            self.depth_files = sorted([os.path.join(depth_dir, f) for f in os.listdir(depth_dir)
                                       if f.endswith(('.png', '.jpg', '.jpeg'))])
            assert len(self.rgb_files) == len(self.depth_files)

    def __len__(self):
        return len(self.rgb_files)

    def __getitem__(self, idx):
        img = Image.open(self.rgb_files[idx]).convert("RGB")
        img = torch.from_numpy(np.array(img)).float().permute(2,0,1) / 255.0

        if self.depth_files:
            depth = Image.open(self.depth_files[idx])
            depth = np.array(depth).astype(np.float32)
            if depth.ndim == 3:
                depth = depth[:,:,0]
            depth[depth < 1] = 1.0
            depth = np.log(depth)
            depth = torch.from_numpy(depth).unsqueeze(0)
            return img, depth
        else:
            return img

# ==========================
# CHEMINS TEST
# ==========================
rgb_dir_test = "/content/drive/MyDrive/imageRGB"
depth_dir_test = "/content/drive/MyDrive/cartePro"  # ou None si pas de vérité

test_dataset = DepthDatasetTest(rgb_dir_test, depth_dir_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ==========================
# CHARGEMENT DU MODELE FINE-TUNÉ
# ==========================
processor = GLPNImageProcessor.from_pretrained("vinvino02/glpn-nyu")
model = GLPNForDepthEstimation.from_pretrained("vinvino02/glpn-nyu")

model.load_state_dict(torch.load("/content/drive/MyDrive/best_glpn.pth", map_location=device))
model.to(device)
model.eval()

# ==========================
# DOSSIER DE SAUVEGARDE
# ==========================
output_dir = "/content/drive/MyDrive/mem"
os.makedirs(output_dir, exist_ok=True)

# ==========================
# BOUCLE DE TEST
# ==========================
absrel_list, delta1_list, delta2_list, delta3_list, accuracy_list = [], [], [], [], []

with torch.no_grad():
    for i, batch in enumerate(tqdm(test_loader, desc="TEST")):
        if len(batch) == 2:
            imgs, depths = batch
            depths = depths.to(device)
        else:
            imgs = batch
            depths = None

        imgs = imgs.to(device)

        # Prétraitement GLPN
        imgs_pil = [Image.fromarray((img.permute(1,2,0).cpu().numpy()*255).astype(np.uint8))
                    for img in imgs]
        pixel_values = processor(images=imgs_pil, return_tensors="pt").pixel_values.to(device)

        # Prédiction
        pred = model(pixel_values=pixel_values).predicted_depth  # [B,1,H,W] normalement

# Vérifie et corrige si nécessaire
        if pred.dim() == 3:  # [B, H, W] ou [B, 1, H]
          pred = pred.unsqueeze(1)  # toujours [B,1,H,W]

        pred = F.interpolate(pred, size=(imgs.shape[2], imgs.shape[3]),
                     mode="bilinear", align_corners=False)


        # Délog et métriques
        if depths is not None:
            pred_real = torch.exp(pred)
            depths_real = torch.exp(depths)

            mask = (depths_real > 0.1) & torch.isfinite(depths_real) & torch.isfinite(pred_real)
            pred_valid = pred_real[mask]
            depth_valid = depths_real[mask]

            # AbsRel
            absrel_list.append(torch.mean(torch.abs(pred_valid - depth_valid) / depth_valid).item())

            # Delta metrics
            ratio = torch.max(pred_valid / depth_valid, depth_valid / pred_valid)
            delta1_list.append((ratio < 1.25).float().mean().item())
            delta2_list.append((ratio < 1.25**2).float().mean().item())
            delta3_list.append((ratio < 1.25**3).float().mean().item())

            # Accuracy spatiale
            threshold = 0.10
            accuracy_list.append((torch.abs(pred_valid - depth_valid) < threshold).float().mean().item())

        # Sauvegarde cartes
        for j, p in enumerate(pred):
            # Délog et CPU
            p_img = torch.exp(p.squeeze(0)).cpu().numpy()
            p_img[~np.isfinite(p_img)] = 0

            # --- Sauvegarde valeurs réelles ---
            np.save(os.path.join(output_dir, f"depth_real_{i*batch_size + j}.npy"), p_img)

            # --- Normalisation pour visualisation avec percentiles ---
            vmin, vmax = np.percentile(p_img, 2), np.percentile(p_img, 98)
            p_norm = np.clip((p_img - vmin) / (vmax - vmin + 1e-8), 0, 1)

            # --- Colormap ---
            p_colored = (plt.get_cmap('magma')(p_norm)[:, :, :3] * 255).astype(np.uint8)
            fname = os.path.basename(test_dataset.rgb_files[i*batch_size + j])
            Image.fromarray(p_colored).save(os.path.join(output_dir, fname))

# ==========================
# AFFICHAGE MÉTRIQUES
# ==========================
if absrel_list:
    print(f"\n✅ Test Metrics:")
    print(f"AbsRel: {np.mean(absrel_list):.4f}")
    print(f"δ1: {np.mean(delta1_list):.4f}, δ2: {np.mean(delta2_list):.4f}, δ3: {np.mean(delta3_list):.4f}")
if accuracy_list:
    print(f"Accuracy spatiale (±0.10): {np.mean(accuracy_list)*100:.2f}%")

print(f"\n✅ Cartes de profondeur sauvegardées dans : {output_dir}")

# ==========================
# AFFICHAGE ALÉATOIRE
# ==========================
num_examples = 5
indices = random.sample(range(len(test_dataset)), num_examples)

for i in indices:
    # Image RGB
    rgb_img = Image.open(test_dataset.rgb_files[i]).convert("RGB")

    # Depth réel
    if test_dataset.depth_files:
        depth_real = np.array(Image.open(test_dataset.depth_files[i]))
    else:
        depth_real = None

    # Depth prédit (.npy)
    import matplotlib.pyplot as plt
    import numpy as np
    from PIL import Image

# p_img : numpy array float de profondeur réelle
    vmin, vmax = np.percentile(p_img, 2), np.percentile(p_img, 98)
    p_norm = np.clip((p_img - vmin) / (vmax - vmin + 1e-8), 0, 1)

# Colormap
    p_colored = (plt.get_cmap('magma')(p_norm)[:, :, :3] * 255).astype(np.uint8)
    Image.fromarray(p_colored).save("/content/drive/MyDrive/mem/test_vis.png")


    # Affichage
    plt.figure(figsize=(15,5))

    plt.subplot(1,3,1)
    plt.imshow(rgb_img)
    plt.title("RGB")
    plt.axis("off")

    plt.subplot(1,3,2)
    if depth_real is not None:
        plt.imshow(depth_real, cmap="inferno")
        plt.title("Depth Réelle")
    else:
        plt.text(0.5, 0.5, "Depth réel non dispo", ha='center', va='center')
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(depth_pred_norm, cmap="inferno")
    plt.title("Depth Prédit")
    plt.axis("off")

    plt.show()


In [ ]:
import os
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from transformers import GLPNForDepthEstimation, GLPNImageProcessor
from tqdm import tqdm

# ==========================
# DEVICE ET BATCH
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 4

# ==========================
# DATASET TEST
# ==========================
class DepthDatasetTest(Dataset):
    def __init__(self, rgb_dir, depth_dir=None):
        self.rgb_files = sorted([os.path.join(rgb_dir, f) for f in os.listdir(rgb_dir)
                                 if f.endswith(('.png', '.jpg', '.jpeg'))])
        self.depth_files = None
        if depth_dir:
            self.depth_files = sorted([os.path.join(depth_dir, f) for f in os.listdir(depth_dir)
                                       if f.endswith(('.png', '.jpg', '.jpeg'))])
            assert len(self.rgb_files) == len(self.depth_files)

    def __len__(self):
        return len(self.rgb_files)

    def __getitem__(self, idx):
        img = Image.open(self.rgb_files[idx]).convert("RGB")
        img = torch.from_numpy(np.array(img)).float().permute(2,0,1) / 255.0

        if self.depth_files:
            depth = Image.open(self.depth_files[idx])
            depth = np.array(depth).astype(np.float32)
            if depth.ndim == 3:
                depth = depth[:,:,0]
            depth[depth < 1] = 1.0
            depth = np.log(depth)  # stockage log pour le modèle
            depth = torch.from_numpy(depth).unsqueeze(0)
            return img, depth
        else:
            return img

# ==========================
# CHEMINS TEST
# ==========================
rgb_dir_test = "/content/drive/MyDrive/imageRGB"
depth_dir_test = "/content/drive/MyDrive/cartePro"  # ou None si pas de vérité

test_dataset = DepthDatasetTest(rgb_dir_test, depth_dir_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ==========================
# CHARGEMENT DU MODELE FINE-TUNÉ
# ==========================
processor = GLPNImageProcessor.from_pretrained("vinvino02/glpn-nyu")
model = GLPNForDepthEstimation.from_pretrained("vinvino02/glpn-nyu")

# Charger checkpoint fine-tuné
model.load_state_dict(torch.load("/content/drive/MyDrive/best_glpn.pth", map_location=device))
model.to(device)
model.eval()

# ==========================
# BOUCLE DE TEST
# ==========================
absrel_list = []
delta1_list = []
delta2_list = []
delta3_list = []
accuracy_list = []  # <- À ajouter !
output_dir = "/content/drive/MyDrive/mem"
with torch.no_grad():
    for i, batch in enumerate(tqdm(test_loader, desc="TEST")):

        if len(batch) == 2:
            imgs, depths = batch
            depths = depths.to(device)
        else:
            imgs = batch
            depths = None

        imgs = imgs.to(device)

        # --- Prétraitement GLPN ---
        imgs_pil = [Image.fromarray((img.permute(1,2,0).cpu().numpy()*255).astype(np.uint8))
                    for img in imgs]
        pixel_values = processor(images=imgs_pil, return_tensors="pt").pixel_values.to(device)

        # --- Prédiction ---
        pred = model(pixel_values=pixel_values).predicted_depth   # [B,1,h,w]
        if pred.dim() == 3:
            pred = pred.unsqueeze(1)

        pred = F.interpolate(pred, size=(imgs.shape[2], imgs.shape[3]),
                             mode="bilinear", align_corners=False)

        if depths is not None:

            # Délog
            pred_real = torch.exp(pred)
            depths_real = torch.exp(depths)

            # --- Masque pixels valides ---
            mask = (depths_real > 0.1) & torch.isfinite(depths_real) & torch.isfinite(pred_real)

            pred_valid = pred_real[mask]
            depth_valid = depths_real[mask]

            # --- AbsRel ---
            absrel = torch.mean(torch.abs(pred_valid - depth_valid) / depth_valid)
            absrel_list.append(absrel.item())

            # --- Delta metrics ---
            ratio = torch.max(pred_valid / depth_valid, depth_valid / pred_valid)
            delta1_list.append((ratio < 1.25).float().mean().item())
            delta2_list.append((ratio < 1.25**2).float().mean().item())
            delta3_list.append((ratio < 1.25**3).float().mean().item())

            # --- Accuracy spatiale (pixel-wise) ---
            threshold = 0.25   # en mètres
            acc = (torch.abs(pred_valid - depth_valid) < threshold).float().mean()
            accuracy_list.append(acc.item())

        # --- Sauvegarde des cartes prédites ---
        import matplotlib.pyplot as plt

        for j, p in enumerate(pred):
              # Délog
            p_img = torch.exp(p.squeeze(0).cpu())
            p_img[~torch.isfinite(p_img)] = 0

              # --- Sauvegarde des valeurs réelles ---
            np.save(os.path.join(output_dir, f"depth_real_{i*batch_size + j}.npy"), p_img.numpy())

              # --- Normalisation pour visualisation ---
            p_norm = (p_img - p_img.min()) / (p_img.max() - p_img.min() + 1e-8)

              # --- Colormap pour mieux voir les variations ---
            p_colored = (plt.get_cmap('magma')(p_norm.numpy())[:, :, :3] * 255).astype(np.uint8)
            Image.fromarray(p_colored).save(os.path.join(output_dir, f"depth_vis_{i*batch_size + j}.png"))



# ==========================
# AFFICHAGE MÉTRIQUES
# ==========================
output_dir = "/content/drive/MyDrive/mem"
if absrel_list:
    print(f"\n✅ Test Metrics:")
    print(f"AbsRel: {np.mean(absrel_list):.4f}")
    print(f"δ1: {np.mean(delta1_list):.4f}, δ2: {np.mean(delta2_list):.4f}, δ3: {np.mean(delta3_list):.4f}")
if accuracy_list:
    print(f"Accuracy spatiale (±0.10): {np.mean(accuracy_list)*100:.2f}%")

print(f"\n✅ Cartes de profondeur sauvegardées dans : {output_dir}")


In [ ]:
import random
import matplotlib.pyplot as plt
# Sélectionner 5 indices aléatoires différents de ceux déjà affichés
num_examples = 5
indices = random.sample(range(len(test_dataset)), num_examples)

for i in indices:
    # Image RGB
    rgb_img = Image.open(test_dataset.rgb_files[i]).convert("RGB")

    # Carte de profondeur réelle
    if test_dataset.depth_files:
        depth_real = np.array(Image.open(test_dataset.depth_files[i]))
    else:
        depth_real = None

    # Carte prédite
    # Charger la carte prédite avec les vraies valeurs float
    pred_path_npy = os.path.join("/content/drive/MyDrive/mem",
                             f"depth_real_{i}.npy")  # correspond à la sauvegarde .npy
    depth_pred = np.load(pred_path_npy)


    # Affichage
    plt.figure(figsize=(15,5))

    plt.subplot(1,3,1)
    plt.imshow(rgb_img)
    plt.title("RGB")
    plt.axis("off")

    plt.subplot(1,3,2)
    if depth_real is not None:
        plt.imshow(depth_real, cmap="inferno")
        plt.title("Depth Réelle")
    else:
        plt.text(0.5, 0.5, "Depth réel non dispo", ha='center', va='center')
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(depth_pred, cmap="inferno")
    plt.title("Depth Prédit")
    plt.axis("off")

    plt.show()


In [ ]:
import random
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image

# ----------------------------
# Fonction de normalisation robuste
# ----------------------------
def normalize_depth(depth):
    p2 = np.percentile(depth, 2)
    p98 = np.percentile(depth, 98)
    depth = np.clip(depth, p2, p98)
    depth = (depth - p2) / (p98 - p2 + 1e-8)
    return depth

# ----------------------------
# Sélection aléatoire
# ----------------------------
num_examples = 5
indices = random.sample(range(len(test_dataset)), num_examples)

for i in indices:
    # RGB
    rgb_img = Image.open(test_dataset.rgb_files[i]).convert("RGB")

    # Depth réelle
    if test_dataset.depth_files:
        depth_real = np.array(Image.open(test_dataset.depth_files[i]))
    else:
        depth_real = None

    # Depth prédite (fichier PNG)
    pred_path = os.path.join("/content/drive/MyDrive/predicted_depths",
                             os.path.basename(test_dataset.rgb_files[i]))

    depth_pred = np.array(Image.open(pred_path)).astype(np.float32)

    # ----------------------------
    # Nettoyage des points noirs
    # ----------------------------
    depth_pred = cv2.medianBlur(depth_pred, 5)

    # ----------------------------
    # Normalisation pour affichage
    # ----------------------------
    depth_pred_norm = normalize_depth(depth_pred)

    # ----------------------------
    # Affichage
    # ----------------------------
    plt.figure(figsize=(15,5))

    plt.subplot(1,3,1)
    plt.imshow(rgb_img)
    plt.title("RGB")
    plt.axis("off")

    plt.subplot(1,3,2)
    if depth_real is not None:
        plt.imshow(depth_real, cmap="inferno")
        plt.title("Depth Réelle")
    else:
        plt.text(0.5, 0.5, "Depth réel non dispo", ha='center', va='center')
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(depth_pred_norm, cmap="inferno")
    plt.title("Depth Prédit (nettoyé)")
    plt.axis("off")

    plt.show()


texter un nouveau code de test


In [ ]:
import os
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from transformers import GLPNForDepthEstimation, GLPNImageProcessor
from tqdm import tqdm
import matplotlib.pyplot as plt
import random

In [ ]:
class DepthDatasetTest(Dataset):
    def __init__(self, rgb_dir, depth_dir=None):
        self.rgb_files = sorted([os.path.join(rgb_dir, f) for f in os.listdir(rgb_dir)
                                 if f.endswith(('.png', '.jpg', '.jpeg'))])
        self.depth_files = None
        if depth_dir:
            self.depth_files = sorted([os.path.join(depth_dir, f) for f in os.listdir(depth_dir)
                                       if f.endswith(('.png', '.jpg', '.jpeg'))])
            assert len(self.rgb_files) == len(self.depth_files)

    def __len__(self):
        return len(self.rgb_files)

    def __getitem__(self, idx):
        # RGB
        img = Image.open(self.rgb_files[idx]).convert("RGB")
        img_tensor = torch.from_numpy(np.array(img)).float().permute(2,0,1) / 255.0

        if self.depth_files:
            # Carte brute pour affichage
            depth_raw = np.array(Image.open(self.depth_files[idx])).astype(np.float32)
            if depth_raw.ndim == 3:
                depth_raw = depth_raw[:,:,0]  # prendre un canal si RGB

            # Carte transformée pour le modèle
            depth = depth_raw.copy()
            depth[depth < 1] = 1.0
            depth = np.log(depth)
            depth_tensor = torch.from_numpy(depth).unsqueeze(0)

            return img_tensor, depth_tensor, depth_raw  # <-- depth_raw est la vraie carte
        else:
            return img_tensor


In [ ]:
rgb_dir_test = "/content/drive/MyDrive/imageRGB"
depth_dir_test = "/content/drive/MyDrive/cartePro"  # ou None si pas de vérité

test_dataset = DepthDatasetTest(rgb_dir_test, depth_dir_test)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

In [ ]:
processor = GLPNImageProcessor.from_pretrained("vinvino02/glpn-nyu")
model = GLPNForDepthEstimation.from_pretrained("vinvino02/glpn-nyu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load("/content/drive/MyDrive/best_glpn.pth", map_location=device))
model.to(device)
model.eval()

In [ ]:
import torch
import numpy as np

########################################
# 1. MODE ÉVALUATION
########################################
model.eval()
criterion = torch.nn.MSELoss()

test_losses = []
test_outputs = []
test_labels = []

########################################
# 2. BOUCLE DE TEST
########################################
for test_inputs, test_labels_batch, depth_raw in test_loader:
    test_inputs = test_inputs.to(device)
    test_labels_batch = test_labels_batch.to(device)

    # Forward
    with torch.no_grad():
        preds = model(pixel_values=test_inputs).predicted_depth

    # Ajuster la forme des prédictions
    if preds.ndim == 3:
        preds = preds.unsqueeze(1)
    elif preds.ndim == 2:
        B = preds.shape[0]
        H, W = test_labels_batch.shape[-2:]
        preds = preds.view(B, 1, H, W)

    # Redimensionner à la taille du GT
    preds = torch.nn.functional.interpolate(
        preds,
        size=test_labels_batch.shape[-2:],
        mode="bilinear",
        align_corners=False
    )

    # Loss et stockage
    loss = criterion(preds, test_labels_batch)
    test_losses.append(loss.item())
    test_outputs.append(preds.detach())
    test_labels.append(test_labels_batch.detach())

########################################
# 3. CONCATÉNATION
########################################
test_outputs = torch.cat(test_outputs).cpu().numpy()
test_labels = torch.cat(test_labels).cpu().numpy()

########################################
# 4. MASQUE DES PIXELS VALIDES
########################################
mask = test_labels > 0.01  # pixels valides
valid_outputs = test_outputs[mask]
valid_labels = test_labels[mask]

########################################
# 5. MASQUE SPATIAL
########################################
# Exemple : pixels à l’intérieur du champ ou zone d’intérêt
# Ici, on peut définir un masque simple basé sur la profondeur non nulle
spatial_mask = (test_labels > 0.01) & (test_labels < 10.0)  # ajuster seuil max selon ton dataset
spatial_outputs = test_outputs[spatial_mask]
spatial_labels = test_labels[spatial_mask]

ratio_map_spatial = np.maximum(
    spatial_outputs / (spatial_labels + 1e-8),
    spatial_labels / (spatial_outputs + 1e-8)
)

spatial_accuracy = np.mean(ratio_map_spatial < 1.25)

########################################
# 6. δ-MÉTRIQUES
########################################
ratio_map = np.maximum(
    valid_outputs / (valid_labels + 1e-8),
    valid_labels / (valid_outputs + 1e-8)
)

delta1 = np.mean(ratio_map < 1.25)
delta2 = np.mean(ratio_map < 1.25**2)
delta3 = np.mean(ratio_map < 1.25**3)

########################################
# 7. AUTRES MÉTRIQUES
########################################
mse = np.mean((valid_outputs - valid_labels) ** 2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(valid_outputs - valid_labels))
abs_rel = np.mean(np.abs(valid_outputs - valid_labels) / (valid_labels + 1e-8))

########################################
# 8. LOSS MOYENNE
########################################
avg_test_loss = np.mean(test_losses)

########################################
# 9. AFFICHAGE
########################################
print("\n📊 RÉSULTATS TEST GLPN")
print(f"Accuracy spatiale : {spatial_accuracy:.4f}")
print(f"Test Loss         : {avg_test_loss:.4f}")
print(f"AbsRel            : {abs_rel:.4f}")
print(f"MSE               : {mse:.6f}")
print(f"RMSE              : {rmse:.4f}")
print(f"MAE               : {mae:.4f}")
print(f"δ1                : {delta1:.4f}")
print(f"δ2                : {delta2:.4f}")
print(f"δ3                : {delta3:.4f}")


In [ ]:
import torch
import numpy as np

########################################
# 1. MODE ÉVALUATION
########################################
model.eval()
criterion = torch.nn.MSELoss()

test_losses = []
test_outputs = []
test_labels = []

########################################
# 2. BOUCLE DE TEST
########################################
for test_inputs, test_labels_batch, depth_raw in test_loader:
    test_inputs = test_inputs.to(device)
    test_labels_batch = test_labels_batch.to(device)

    # Forward
    with torch.no_grad():
        preds = model(pixel_values=test_inputs).predicted_depth

    # Ajuster la forme des prédictions
    if preds.ndim == 3:
        preds = preds.unsqueeze(1)
    elif preds.ndim == 2:
        B = preds.shape[0]
        H, W = test_labels_batch.shape[-2:]
        preds = preds.view(B, 1, H, W)

    # Redimensionner à la taille du GT
    preds = torch.nn.functional.interpolate(
        preds,
        size=test_labels_batch.shape[-2:],
        mode="bilinear",
        align_corners=False
    )

    # Loss et stockage
    loss = criterion(preds, test_labels_batch)
    test_losses.append(loss.item())
    test_outputs.append(preds.detach())
    test_labels.append(test_labels_batch.detach())

########################################
# 3. CONCATÉNATION
########################################
test_outputs = torch.cat(test_outputs).cpu().numpy()
test_labels = torch.cat(test_labels).cpu().numpy()

########################################
# 4. MASQUE DES PIXELS VALIDES
########################################
mask = test_labels > 0.01  # pixels valides
valid_outputs = test_outputs[mask]
valid_labels = test_labels[mask]

########################################
# 5. RATIO POUR δ-MÉTRIQUES
########################################
ratio_map = np.maximum(
    valid_outputs / (valid_labels + 1e-8),
    valid_labels / (valid_outputs + 1e-8)
)

# δ-métriques
delta1 = np.mean(ratio_map < 1.25)
delta2 = np.mean(ratio_map < 1.25**2)
delta3 = np.mean(ratio_map < 1.25**3)

########################################
# 6. PRÉCISION SPATIALE
########################################
# La vraie précision spatiale : proportion de pixels valides correctement localisés
# Ici, on utilise le même ratio < 1.25 mais seulement sur les pixels valides
spatial_accuracy = np.sum(ratio_map < 1.25) / mask.sum()

########################################
# 7. AUTRES MÉTRIQUES
########################################
mse = np.mean((valid_outputs - valid_labels) ** 2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(valid_outputs - valid_labels))
abs_rel = np.mean(np.abs(valid_outputs - valid_labels) / (valid_labels + 1e-8))

########################################
# 8. LOSS MOYENNE
########################################
avg_test_loss = np.mean(test_losses)

########################################
# 9. AFFICHAGE
########################################
print("\n📊 RÉSULTATS TEST GLPN")
print(f"Accuracy spatiale : {spatial_accuracy:.4f}")
print(f"Test Loss         : {avg_test_loss:.4f}")
print(f"AbsRel            : {abs_rel:.4f}")
print(f"MSE               : {mse:.6f}")
print(f"RMSE              : {rmse:.4f}")
print(f"MAE               : {mae:.4f}")
print(f"δ1                : {delta1:.4f}")
print(f"δ2                : {delta2:.4f}")
print(f"δ3                : {delta3:.4f}")


In [ ]:
import torch
import numpy as np

########################################
# 1. MODE ÉVALUATION
########################################

model.eval()
criterion = torch.nn.MSELoss()

test_losses = []
test_outputs = []
test_labels = []

########################################
# 2. BOUCLE DE TEST
########################################
for test_inputs, test_labels_batch, depth_raw in test_loader:
    test_inputs = test_inputs.to(device)
    test_labels_batch = test_labels_batch.to(device)

    # Forward
    with torch.no_grad():
        preds = model(pixel_values=test_inputs).predicted_depth

    # Corriger forme prédiction
    if preds.ndim == 3:
        preds = preds.unsqueeze(1)
    elif preds.ndim == 2:
        B = preds.shape[0]
        H, W = test_labels_batch.shape[-2:]
        preds = preds.view(B, 1, H, W)

    # Resize à la taille du GT
    preds = torch.nn.functional.interpolate(
        preds,
        size=test_labels_batch.shape[-2:],
        mode="bilinear",
        align_corners=False
    )

    # Loss et stockage
    loss = criterion(preds, test_labels_batch)
    test_losses.append(loss.item())
    test_outputs.append(preds.detach())
    test_labels.append(test_labels_batch.detach())


########################################
# 3. CONCATÉNATION
########################################

test_outputs = torch.cat(test_outputs).cpu().numpy()
test_labels = torch.cat(test_labels).cpu().numpy()

########################################
# 4. MASQUE DES PIXELS VALIDES
########################################

mask = test_labels > 0.01
test_outputs = test_outputs[mask]
test_labels = test_labels[mask]

ratio_map = np.maximum(
    test_outputs / (test_labels + 1e-8),
    test_labels / (test_outputs + 1e-8)
)

spatial_accuracy = (ratio_map < 1.25).astype(np.float32)


########################################
# 5. MÉTRIQUES DEPTH ESTIMATION
########################################

mse = np.mean((test_outputs - test_labels) ** 2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(test_outputs - test_labels))
abs_rel = np.mean(np.abs(test_outputs - test_labels) / (test_labels + 1e-8))

ratio = np.maximum(
    test_outputs / (test_labels + 1e-8),
    test_labels / (test_outputs + 1e-8)
)

delta1 = np.mean(ratio < 1.25)
delta2 = np.mean(ratio < 1.25 ** 2)
delta3 = np.mean(ratio < 1.25 ** 3)

########################################
# 6. LOSS MOYENNE
########################################

avg_test_loss = np.mean(test_losses)

########################################
# 7. AFFICHAGE
########################################

print("\n📊 RÉSULTATS TEST GLPN")
print("Spatial δ1 accuracy:", spatial_accuracy.mean())
print(f"Test Loss : {avg_test_loss:.4f}")
print(f"AbsRel    : {abs_rel:.4f}")
print(f"MSE       : {mse:.6f}")
print(f"RMSE      : {rmse:.4f}")
print(f"MAE       : {mae:.4f}")
print(f"δ1        : {delta1:.4f}")
print(f"δ2        : {delta2:.4f}")
print(f"δ3        : {delta3:.4f}")


In [ ]:
print("GT shape:", depth_raw.shape)
print("Pred shape:", pred_resized.shape)
print("RGB shape:", rgb.shape)


In [ ]:
idx = 0
img, depth_tensor, depth_raw = test_dataset[idx]

# Normalisation pour affichage
gt_vis = (depth_raw - depth_raw.min()) / (depth_raw.max() - depth_raw.min() + 1e-8)

plt.imshow(gt_vis, cmap="inferno")
plt.title("Carte de profondeur réelle (raw)")
plt.colorbar()
plt.show()
